# Section 3 — Freeze-Boundary Study ⭐

> Design: `PIPELINE.md`. Needs: Section 2's feature cache.

## The question

The model is `ResNet50 → LSTM → head`. The ResNet is frozen forever after Section 1.
**Should the LSTM be frozen too?**

```
PROBE A     [16x2048] --> LSTM (locked)  --> head (learns)        70,416 train
PROBE B     [16x2048] --> LSTM (learns)  --> head (learns)     2,431,760 train
```

Both load the **same** Task-0 LSTM weights. The only difference is `requires_grad`.

## Method

Train on **all classes at once** — no tasks, no waves, no forgetting. Just: *how good
can this possibly get?*

## Two results from one experiment

**1. The architecture decision.**

| Result | Meaning | Use |
|---|---|---|
| B >> A | the LSTM was the bottleneck | **B** — replay stores `(16,2048)`, ~960 KB/class |
| B ~= A | frozen 256-d features already suffice | **A** — replay stores `256-d`, ~15 KB/class |

**2. THE CEILING** — which matters more.

```
"SmartReplay = 71%"                    <- means nothing
"SmartReplay = 71%, ceiling = 85%"     <- means 71 out of a possible 85
```

Every number in Sections 4-10 is read against this.

## What to expect

Base classes are all full-body sports (Diving, PoleVault, HorseRace...). The new classes
are close-up faces (`ApplyEyeMakeup`, `ApplyLipstick`) and fine-grained
(`BoxingPunchingBag` vs `BoxingSpeedBag`). A 256-d bottleneck trained on sports may have
discarded the detail those need — which is exactly what probe A would reveal.

## 1 — Bootstrap

In [ ]:
# ── ULAL bootstrap — self-contained, no Drive dependency ────────────────────
# One shallow git clone instead of hundreds of API calls. subprocess takes a
# LIST, never a shell string, so the '&' in the branch name is safe.
import os, sys, subprocess
from getpass import getpass

REPO, BRANCH, ROOT = "silvergjeka22/Unified-Lifelong-Action-Learning", "GAN&ActiveLearning", "/content/ulal"
token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token (hidden): ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(f"{ROOT}/.git"):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)

sys.path.insert(0, ROOT)
print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
# ── Drive + dataset + paths ─────────────────────────────────────────────────
# Exports ULAL_* env vars that config.py reads. Nothing is written to disk and
# no kernel restart is needed.
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

In [ ]:
# ── Global label space (src/data/cache.py) ──────────────────────────────────
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]     # Sections 4-5 stream over these
FEWSHOT_TASK = 3          # Section 7 probe only

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

## 2 — Load cached features

In [ ]:
# ── Load cached frame features (src/data/cache.py) ──────────────────────────
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

## 3 — Joint training data

**All** classes, all tasks, one pot. No CL here.

In [ ]:
JOINT_EPOCHS = 20
LR, WD, BATCH, LS = 5e-4, 0.03, 32, 0.10

T_LAST = len(TASK_CLASSES) - 1
jt_s, jt_y   = cat_upto(CACHE, T_LAST, "train")
jv_s, jv_y   = cat_upto(CACHE, T_LAST, "val")
jte_s, jte_y = cat_upto(CACHE, T_LAST, "test")
N_ALL = NUM_AFTER[-1]

print(f"Joint train {tuple(jt_s.shape)}")
print(f"      val   {tuple(jv_s.shape)}")
print(f"      test  {tuple(jte_s.shape)}")
print(f"{N_ALL} classes, {len(set(jt_y.tolist()))} present in train")

## 4 — Run both probes

Same data, optimiser, epochs. Only the LSTM lock differs.

In [ ]:
freeze_study = {}

for key, label, freeze in [("A", "LSTM FROZEN    (head only)", True),
                           ("B", "LSTM TRAINABLE (lstm+head)", False)]:
    print("=" * 70); print(f"PROBE {key} — {label}"); print("=" * 70)

    head = make_temporal_head(N_ALL, freeze_lstm=freeze,
                              lstm_state=LSTM_STATE, device=device)
    n_tr = head.trainable_params()
    print(f"  trainable: {n_tr:,}")

    head = finetune_head(
        head, jt_s, None, jt_y, jv_s, jv_y, device,       # train_t=None -> MSE skipped
        epochs=JOINT_EPOCHS, lr=LR, wd=WD, batch_size=BATCH,
        patience=cfg.FINETUNE_PATIENCE,
        ce_weight=1.0, mse_weight=0.0, label_smoothing=LS,
    )

    acc = eval_head(head, jte_s, jte_y, device)
    freeze_study[key] = {"label": label.strip(), "joint_test_acc": acc,
                         "trainable": n_tr, "freeze_lstm": freeze}
    print(f"  -> JOINT TEST ACCURACY: {acc:.2%}\n")

    torch.save(head.state_dict(), f"{cfg.CKPT_DIR}/head_joint_probe{key}.pt")
    del head
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5 — The decision

In [ ]:
a, b = freeze_study["A"], freeze_study["B"]
gap  = b["joint_test_acc"] - a["joint_test_acc"]

print("=" * 70); print("  FREEZE-BOUNDARY STUDY"); print("=" * 70)
print(f"  A  LSTM frozen     : {a['joint_test_acc']:>7.2%}   ({a['trainable']:>9,} trainable)")
print(f"  B  LSTM trainable  : {b['joint_test_acc']:>7.2%}   ({b['trainable']:>9,} trainable)")
print(f"  gap (B - A)        : {gap:>+7.2%}")
print(f"  cost of B          : {b['trainable']/a['trainable']:>7.0f}x more trainable params")
print("=" * 70)

FREEZE_LSTM = gap <= 0.05
CEILING     = max(a["joint_test_acc"], b["joint_test_acc"])

if FREEZE_LSTM:
    print("  => A is within 5pp of B.")
    print("     The frozen 256-d features already hold the new classes.")
    print("     BOUNDARY A: ResNet[frozen] -> LSTM[frozen] -> head[train]")
    print("     Replay unit = 256-d vectors (~15 KB/class). Cheaper, and honest.")
else:
    print("  => B wins by more than 5pp. The LSTM WAS the bottleneck.")
    print("     BOUNDARY B: ResNet[frozen] -> LSTM[train] -> head[train]")
    print("     Replay unit = (16,2048) sequences (~960 KB/class).")

print(f"\n  CEILING = {CEILING:.2%}")
print("  Read every Section 4-10 number against this, never in isolation.")

## 6 — Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = ["A\nLSTM frozen", "B\nLSTM trainable"]
accs  = [a["joint_test_acc"], b["joint_test_acc"]]
trs   = [a["trainable"], b["trainable"]]

axes[0].bar(names, accs, color=["#94A3B8", "#2563EB"], width=0.55)
axes[0].axhline(CEILING, ls="--", c="#16A34A", lw=1.5, label=f"ceiling {CEILING:.1%}")
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.02, f"{v:.1%}", ha="center", fontweight="bold")
axes[0].set_ylim(0, 1.1); axes[0].set_ylabel("Joint test accuracy")
axes[0].set_title("How good can it get?\n(all classes at once)", fontsize=11)
axes[0].legend(); axes[0].grid(alpha=0.3, axis="y")

axes[1].bar(names, trs, color=["#94A3B8", "#DC2626"], width=0.55)
axes[1].set_yscale("log"); axes[1].set_ylabel("Trainable params (log)")
axes[1].set_title("What it costs", fontsize=11)
for i, v in enumerate(trs):
    axes[1].text(i, v * 1.15, f"{v:,}", ha="center", fontsize=9, fontweight="bold")
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
plt.savefig(f"{cfg.RESULTS_DIR}/stage0_freeze_study.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 — Save

In [ ]:
out = {
    "stage": "0_freeze_boundary",
    "probes": freeze_study,
    "gap": gap,
    "chosen_freeze_lstm": bool(FREEZE_LSTM),
    "ceiling": CEILING,
    "num_classes": int(N_ALL),
    "config": {"epochs": JOINT_EPOCHS, "lr": LR, "wd": WD,
               "batch": BATCH, "label_smoothing": LS, "seed": cfg.SEED},
}
with open(f"{cfg.RESULTS_DIR}/stage0_freeze_study.json", "w") as f:
    json.dump(out, f, indent=2)

print(json.dumps(out, indent=2))
print(f"\nSaved -> {cfg.RESULTS_DIR}/stage0_freeze_study.json")
print("Section 4 reads FREEZE_LSTM and CEILING from this file.")

---

## Done — next: Section 4

**Read your result**

- **Gap > 5pp** → the LSTM was blocking. Boundary B is justified *by evidence*, not
  assumption. Replay stores sequences.
- **Gap ≤ 5pp** → frozen features were fine all along. Take A: 34x fewer trainable params
  and 64x cheaper replay for almost the same accuracy. That is a **good** result, not a
  disappointing one.

Either answer is publishable — the point was to stop guessing.

**Sanity:** if both probes sit near chance (~1/N), the teacher checkpoint didn't load —
check `cfg.RESNET50_PATH`.

**Next:** `4_continual_learning/4.0_cl_comparison.ipynb` — classes arrive in waves, four
strategies compete, everything measured against the ceiling you just computed.